In [ ]:
if you use vs code then plz create your environment first so that your project don't get clash with any another project
use --> (uv .venv)command and for activate use( source .venv/bin/activate)
#  requirement.txt
langchain,langchain-core,langchain-community,
# llm providers 
langchain-openai
langchain-mistralai
openai
mistralai
# vector Database
chormadb
# embedding models
sentence-transformers
# document Loaders dependencies

# text preprocessing
tiktoken
# Environment Variables
python-dotenv
# data handling
pandas
numpy
# api/applayer (optional but common in RAG apps)
fastapi
uvicorn
# UI (optional)
streamlit
# Utilities
tqdm
requests
# how to install
uv pip install dotenv

In [ ]:
MISTRAL_API_KEY ="cS34y6HaFuvnS5GDgtPoQQESMQkYWZlm"


In [ ]:
# document loaders
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI
from langchain_community.document_loaders import TextLoader,PyPDFLoader,WebBaseLoader

load_dotenv()
data= TextLoader('document loaders/notes.txt') # this provide list which include [metadata() , pagecontent()]
docs = data.load()
dataPDF= PyPDFLoader('document loaders/GRU.pdf') # this provide list which include [metadata() , pagecontent()]
pdfDocs = dataPDF.load()
webData= WebBaseLoader('https://www.apple.com/in/') # this provide list which include [metadata() , pagecontent()]
WebDocs = webData.load()

template= ChatPromptTemplate.from_messages(
    [('system',"you are a AI that summarize the text"),
    ("human",{"data"})]
)
model = ChatMistralAI(model = 'mistral-small-2506')
prompt = template.format_messages(data= docs[0].page_content)
PdfPrompt = template.format_messages(data= pdfDocs[0].page_content)
WebPrompt = template.format_messages(data= WebDocs[0].page_content)
result = model.invoke(prompt,PdfPrompt,WebPrompt)
print(result.content) 



In [ ]:
TextSplitter-->most of the embedding models and language model have a context window and they cannot take infinite tokens
at the same time so for that we have to use text splitting
2.Better Retrieval in RAG --> in Retrival-Augmented Generation(RAG), we search through vector embeddings.
if the documents is too large:
    embedding becomes less precise
smaller chunks allow the system to retrieve only the most relevant piece of information instead of the whole document.
3.more Accurate Embeddings-->embeddings work best when the text represents one clear idea or topic.
if you embed very large text:
    .multiple topics mix together
    .semantic meaning becomes blurred
Chunking ensures each embedding represents a focused concept.
*types of text splitting
1. character-based splitting
2.token-based splitting
3.semantic/meaning-based splitting

In [ ]:
# character-based splitting example
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator ='',
    chunk_size =1000,
    chunk_overlap=1
)
data = TextLoader('document loaders/notes.txt')
docs = data.load()
chunks = splitter.split_documents(docs)
for i in chunks:
    print(i.page_content)


In [ ]:
# token based spltting  (most use splitter method)
from langchain_community.document_loaders import TextLoader,PyPDFLoader,WebBaseLoader
from langchain_text_splitters import TokenTextSplitter


splitter =TokenTextSplitter(
    chunk_size =100,
    chunk_overlap=10
)
data = PyPDFLoader('document loaders/notes.txt')
docs = data.load()

chunks = splitter.split_documents(docs)
print(len(chunks))

In [ ]:
 # splitting recursively text splitter ['\n\n','\n'," ",""] it check whhich splitter is more accurate according to chunks
from langchain_community.document_loaders import TextLoader #better than other two
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separator ='',
    chunk_size =1000,
    chunk_overlap=1
)
data = TextLoader('document loaders/notes.txt')
docs = data.load()
chunks = splitter.split_documents(docs)
for i in chunks:
    print(i.page_content)

In [ ]:
TEXTSPLITTER-> DeepLearning is a specialized area of 'ml' that uses neural networks with multiple layers to automatically
learn complex representations from large amounts of data. Natural Language Processign(NLP) is another important area of artificial intelligence
that focuses on enabling computers to understand, iterpret, and generate human language in tasks such as translation, chatbots, and sentiments analysis

In [ ]:
# 3.semantic/meaning-based splitting (not in use much still underdevelopment)


In [ ]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI
from langchain_community.document_loaders import TextLoader,PyPDFLoader,WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
data= TextLoader('document loaders/notes.txt') 
docs = data.load()
dataPDF= PyPDFLoader('document loaders/GRU.pdf')
pdfDocs = dataPDF.load()
webData= WebBaseLoader('https://www.apple.com/in/')
WebDocs = webData.load()
# the problem here is that if the token limit hits then it can't be execute (to solve this problem we use splitter) 

splitter = RecursiveCharacterTextSplitter(
    separator ='',
    chunk_size =1000,
    chunk_overlap=1
)
chunksDocs = splitter.split_documents(docs)
chunksPdfDocs = splitter.split_documents(pdfDocs)
chunksWebDocs = splitter.split_documents(WebDocs)

# now with splitter help we create chunks but the problem is instead of feeding that much data direct to llm we create storage (vectorDB)

template= ChatPromptTemplate.from_messages(
    [('system',"you are a AI that summarize the text"),
    ("human",{"data"})]
)
model = ChatMistralAI(model = 'mistral-small-2506')
prompt = template.format_messages(data= chunksDocs[0].page_content) 
PdfPrompt = template.format_messages(data= chunksPdfDocs[0].page_content)
WebPrompt = template.format_messages(data= chunksWebDocs[0].page_content)
result = model.invoke(prompt,PdfPrompt,WebPrompt)
print(result.content)

In [ ]:
# VECTOR DATABASE
pdf------>chunks------->embeddings([0.44,0.21,0.67,0.33])------->vectorstore
# now question is we have database like mysql,mongodb etc. so why do we even need vector databases?
suppose in database there is 1 lakh embedding--> how to retireve it (the biggest problem is this 512 dimension embedding is different from all the
1 lakh embedding in our database so we conduct a similarity search with all the 1 lakh embedding)it can become more big so they are not the reliable
option for similarity searching
but in vector data base--> we use KNN algo 
IVF--->(inverted file index) let say we divide our database in 5 cluster using k-means or any other algo.so each will have 20000 embedding and all of
them have some sort of similarity.(after we calulate the average embedding of all the 20000 embeddings. we can call it
centroid. so now we have 5 average embeddings).
NOW for the query embedding we can find which cluster is suitable for searching and we search in those 20000 embeddings. that is actually 5 times faster
than normal database.(like user have query then it don't search all embedding it check which cluster is best and find in those cluster which is 20000)

# difference b/w normal and vector database
data stored - structured data(text,numbers,rows) | embedding(vectors like[0.23,23.3])
search type - exact match | similarity search
query ex    - where movie="KGF" | "movies about gangster"
indexing    - B-tree,Hash | HNSW,IVF,PQ
Use case    - Banking,user records,transactions | Ai search recommendation
Comparison method - Equality or filtering | cosine similarity/ vector

In [ ]:
# HOW to use vector database (implimentation)
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document

docs = [
    Document(page_content="Python is widely used in Artificial Intelligence.", metadata={"source": "AI_book"}),
    Document(page_content="Pandas is used for data analysis in Python.", metadata={"source": "DataScience_book"}),
    Document(page_content="Neural networks are used in deep learning.", metadata={"source": "DL_book"}),
]

embedding_model = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents = docs,
    embedding= embedding_model,
    persist_directory= "chroma-db"
)

result = vectorstore.similarity_search("what is used for data analysis?",k=2)

for r in result:
    print(r.page_content)
    print(r.metadata)

retriver = vectorstore.as_retriever()

docs = retriver.invoke("Explain deep learning")

for d in docs:
    print(d.page_content)


In [ ]:
Retrievers diagram

            vector store-----
               |            |
query-----> Retrievers   similarChunks + Query ---------> prompt--------->LLM

A retriever takes a user query and returns the most relevant documents or chunks from a database.

Now there are 2 types of retirever
1. by data source (wikipedia, Arxiv, PubMed, etc)
2. by retrieval strategy (similarity,MMR, MultiQuery, etc.)

when we talk about retrievers using data sources, we mean retrievers that fetch information from specific external sources or databases instead
of your own vector store. these retrievers connect to APIs or datasets and return relevant documents for the query.

In [1]:
from langchain_community.retrievers import ArxivRetriever

#create the retrievers
retrievers = ArxivRetriever(
    load_max_docs = 2,
    load_all_available_meta = True
)
# query arxiv
docs = retriever.invoke('large language models')

# print results
for i , doc in enumerate(docs):
    print(f"\nResult {i+1}")
    print("title:" doc.metadata.get("Title"))
    print("Authors:" doc.metadata.get("Authors"))
    print("Sumarry:" doc.page_content[:500]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2277621571.py, line 14)

In [ ]:
 Retrievers ----> search strategies - similarity search
the system compares the query vector with document vectors using similarity metrics like:
    . cosine similiarity(most common)
    . Dot product
    . Euclidean Distance
  the retriever finds the most similar vectors.
  Top-k documents are retrieved

Search Strategies - MMR (maximum marginal relevance) -- goal-> 1. relvance to the query, Diversity among retrived docs
Lets understand this with an example:- imagine you are building a RAG system for yours students. your knowldge base contains many chunks about 
Gradient descent . your database chunks look like this:
        chunk1: Gradient descent is an optimizaton algo used in ml
        chunk2: Gradient descent minimizes the loss func.
        chunk3: Gradient descent is an optimization that minimizes the loss func.                                                                    
        chunk4: Neural networks use gradient descent for training.
        chunk5: Support Vector Machines are Supervised learning algo

now the user asks what is gragient descent?
Normal similarity search finds the most similar chunks to the query embedding.
and gives chunk1, chunk2, chunk3

but if you see carefully all the 3 chunks are saying the same thing.
This wastes:
 .context window , .token usage , .information diversity
        the LLM doesn't learn new info.

In [8]:

from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings



docs = [
    Document(page_content="Gradient descent is an optimization algorithm used in machine learning."),
    Document(page_content="Gradient descent minimizes the loss function."),
    Document(page_content="Gradient descent is an optimization that minimizes the loss function."),
    Document(page_content="Neural networks use gradient descent for training."),
    Document(page_content="Support Vector Machines are supervised learning algorithms.")
]


embeddings = HuggingFaceEmbeddings()


vectorstore = Chroma.from_documents(docs, embeddings)


similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

print("\n===== Similarity Search Results =====\n")

similarity_docs = similarity_retriever.invoke("What is gradient descent?")

for doc in similarity_docs:
    print(doc.page_content)


mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

print("\n===== MMR Results =====\n")

mmr_docs = mmr_retriever.invoke("What is gradient descent?")

for doc in mmr_docs:
    print(doc.page_content)

C:\Users\risha\AppData\Local\Temp\ipykernel_4252\60359624.py:16: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== Similarity Search Results =====

Gradient descent is an optimization algorithm used in machine learning.
Gradient descent is an optimization algorithm used in machine learning.
Gradient descent is an optimization algorithm used in machine learning.

===== MMR Results =====

Gradient descent is an optimization algorithm used in machine learning.
Gradient descent minimizes the loss function.
Support Vector Machines are supervised learning algorithms.


In [9]:
# multi query
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv



load_dotenv()

docs = [
    Document(page_content="Gradient descent is an optimization algorithm used in machine learning."),
    Document(page_content="Gradient descent minimizes the loss function."),
    Document(page_content="Gradient descent is an optimization that minimizes the loss function."),
    Document(page_content="Neural networks use gradient descent for training."),
    Document(page_content="Support Vector Machines are supervised learning algorithms.")
]


embeddings = HuggingFaceEmbeddings()

vectorstore = Chroma.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()


llm = ChatMistralAI(model="mistral-small-latest")

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

query = "What is gradient descent?"

docs = multi_query_retriever.invoke(query)


print("\nRetrieved Documents:\n")

for doc in docs:
    print(doc.page_content)

C:\Users\risha\AppData\Local\Temp\ipykernel_4252\1300788596.py:22: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HTTPStatusError: Error response 401 while fetching https://api.mistral.ai/v1/chat/completions: {"detail":"Unauthorized"}

In [1]:
# final projectt

from langchain_community.embeddings import HuggingFaceEmbeddings 
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


vectorstore = Chroma(
    persist_directory= "chroma_db",
    embedding_function=embedding_model
)

retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {
        "k" : 4,
        "fetch_k":10,
        "lambda_mult" :0.5
    }
)

llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

#prompt template 
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful AI assistant.Use ONLY the provided context to answer the question.

        If the answer is not present in the context,say: "I could not find the answer in the document.""""
        ),
        
        ( "human","""Context:{context} Question:{question}""" )
    ]
)

print("Rag system created ")

print("press 0 to exit ")

while True:
    query = input("You : ")
    if query == "0":
        break 
    
    docs = retriever.invoke(query)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )
    
    final_prompt = prompt.invoke({
        "context" :context,
        "question": query
    })
    
    response = llm.invoke(final_prompt)

    print(f"\n AI: {response.content}")
    

SyntaxError: unterminated string literal (detected at line 39) (215491040.py, line 39)

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# =====================================================================
# 1. SETUP ENVIRONMENT & LLM
# =====================================================================
print("🔄 Setting up Mistral LLM...")
# Using the API key from your notebook cell
os.environ["MISTRAL_API_KEY"] = "cS34y6HaFuvnS5GDgtPoQQESMQkYWZlm"

# Initialize the Mistral Large Language Model
llm = ChatMistralAI(model="mistral-medium")

# =====================================================================
# 2. LOAD AND SPLIT THE DOCUMENT
# =====================================================================
print("📄 Loading and splitting document...")
# Load your private company document
loader = TextLoader("sample_document.txt")
documents = loader.load()

# Split the document into small, overlapping chunks so no info is lost
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, 
    chunk_overlap=20
)
chunks = text_splitter.split_documents(documents)
print(f"✅ Split document into {len(chunks)} text chunks.")

# =====================================================================
# 3. CREATE EMBEDDINGS AND STORE IN CHROMADB
# =====================================================================
print("🧠 Generating embeddings and building Vector DB (Chroma)...")
# Download a lightweight, open-source embedding model from HuggingFace
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the vector database in-memory (or give it a path to save on disk)
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embedding_model
)

# Turn the vector database into a "Retriever"
retriever = vector_db.as_retriever(search_kwargs={"k": 1}) 
print("✅ Vector database is ready!")

# =====================================================================
# 4. DEFINE THE RAG PROMPT TEMPLATE
# =====================================================================
# This tells the LLM exactly how to behave with the retrieved context
rag_template = """
You are a helpful company assistant. Answer the question using ONLY the provided context. 
If you do not know the answer based on the context, say "I cannot find that in the company records."

CONTEXT:
{context}

QUESTION: 
{question}

ANSWER:
"""
prompt = ChatPromptTemplate.from_template(rag_template)

# =====================================================================
# 5. ASSEMBLE THE CHAIN AND RUN A QUERY
# =====================================================================
print("\n🔗 Assembling the RAG Chain...")

# Helper function to format retrieved documents nicely for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for docs in docs)

# LangChain Expression Language (LCEL) Chain pipeline
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test our RAG system!
query = "How do I claim my remote work setup money and when is the deadline?"
print(f"\n🙋 User Question: '{query}'")

print("🤖 AI is thinking...")
response = rag_chain.invoke(query)

print("\n✨ AI Response:")
print(response)